In [1]:
import pandas as pd

# INITIALIZATION
# Loading the specified raw data file
file_name = '100-families-data.xlsx'
df = pd.read_excel(file_name)

# MAPPING NEW CATEGORIES IN 'engagement'
# Defining a dictionary mapping the old categories to the new ones
engagement_mapping = {
    'Child Not in School': 'Child',
    'Infant': 'Child',
    'Child out of School': 'Child',
    'Landowner': 'Farmer',
    'Landowner Farmer': 'Farmer',
    'Industrial Labour': 'Private Sector Employee',
    'Private contractual worker': 'Private Sector Employee',
    'State Govt./PSU contractual Employee': 'Private Sector Employee',
    'Senior Citizens Old Age Pensioner': 'Senior Citizen',
    'Super Senior Citizen Social Security Pensioner': 'Senior Citizen',
    'Agricultural labour': 'Labour',
    'Construction worker': 'Labour',
    'Other labour': 'Labour'
}

# Applying the mapping. If a category isn't in the dictionary (like 'Student'), it remains unchanged.
df['engagement'] = df['engagement'].replace(engagement_mapping)

# CREATING 'is<category>' COLUMNS (ONE-HOT ENCODING)
# pd.get_dummies automatically creates binary columns based on categories
engagement_dummies = pd.get_dummies(df['engagement'], prefix='is', prefix_sep='')
engagement_dummies = engagement_dummies.astype(int) # Ensuring output is 1/0 instead of True/False
df = pd.concat([df, engagement_dummies], axis=1)

# HANDLING NULLS & ONE-HOT ENCODE 'memberVerifiedRange'
# Replacing NULL/NaN with '0'
df['memberVerifiedRange'] = df['memberVerifiedRange'].fillna('0').astype(str).str.strip() 

# Creating 'in<category>' columns
# income_dummies = pd.get_dummies(df['memberVerifiedRange'], prefix='in', prefix_sep='')
# income_dummies = income_dummies.astype(int)
# df = pd.concat([df, income_dummies], axis=1)
# Uncomment the above lines to get the separate columns for 'meberVerfiedRange'

# DROPPING & RENAMING COLUMNS
# Dropping specified legacy binary flags
columns_to_drop = ['isGovEmp', 'isLabour', 'isGovPensioner', 'isFarmer']
# Using list comprehension to safely drop only if they exist in the dataframe
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

# Renaming education column
if 'isEducationDataAvailable' in df.columns:
    df = df.rename(columns={'isEducationDataAvailable': 'Private_or_Gov_School'})

# DROPPING UNNECESSARY INDIVIDUAL COLUMNS BEFORE GROUPING
# Dropping member-specific demographic columns that can't be grouped at the family level
member_cols_to_drop = ['hasmemberid', 'qualification', 'age', 'gender', 'engagement', 'memberVerifiedRange']
df = df.drop(columns=[col for col in member_cols_to_drop if col in df.columns])

# AGGREGATING (GROUP BY FAMILY ID)
# Defining which columns need to be SUMMED
# This includes all the newly created dummy columns + the specific proxy flags as requested
columns_to_sum = list(engagement_dummies.columns) + ['isIncomeTaxPay', 'isRuralProperty', 'isUrbanProperty', 'isElectrictyMapping', 'Private_or_Gov_School', 'isFourVehicle'] \
#                 + list(income_dummies.columns)
# Uncomment the above line to get the separate columns for 'memberVerifiedRange'

columns_to_sum = [col for col in columns_to_sum if col in df.columns]

# Defining which columns are shared across the family and just need the FIRST value retained
columns_to_keep_first = ['district', 'RuralUrban', 'familyRange']
columns_to_keep_first = [col for col in columns_to_keep_first if col in df.columns]

# Building the aggregation logic dictionary
aggregation_logic = {col: 'sum' for col in columns_to_sum}
aggregation_logic.update({col: 'first' for col in columns_to_keep_first})

# Executing the group by operation
final_dataset = df.groupby('hasfamilyid', as_index=False).agg(aggregation_logic)

# FINAL EXPORT
final_dataset.to_excel('final_dataset.xlsx', index=False)
print("Data processing complete! Saved to 'final_dataset.xlsx'.")
print(f"Total Families: {len(final_dataset)}")
print(f"Total Columns: {len(final_dataset.columns)}")

Data processing complete! Saved to 'final_dataset.xlsx'.
Total Families: 100
Total Columns: 20
